In [ ]:
pip install protobuf==3.19.6

In [ ]:
pip install tensorflow==2.8.0

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

# Load data
Input_path = '/content/drive/MyDrive/EMG-data.csv'
df = pd.read_csv(Input_path)
features = df.drop(columns=["label","class","time"])
Class = df["class"].values
features = features.values

# Train-test split
x_train, x_test, y_train, y_test = train_test_split(features, Class, test_size=0.2, random_state=1)

# Normalizing data
mean, std = x_train.mean(axis=0), x_train.std(axis=0)
x_train, x_test = (x_train - mean) / std, (x_test - mean) / std

# Reshape data
x_train = np.expand_dims(x_train, axis=1)
x_test = np.expand_dims(x_test, axis=1)
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)

def plot(loss,val_loss,acc,val_acc):
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs = range(1, len(loss) + 1)

    plt.plot(epochs, loss, 'bo', label='Training loss')
    plt.plot(epochs, val_loss, 'b', label='Validation loss')
    plt.title('Training and validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']

    epochs = range(1, len(acc) + 1)

    plt.plot(epochs, acc, 'bo', label='Training acc')
    plt.plot(epochs, val_acc, 'b', label='Validation acc')
    plt.title('Training and validation acc')
    plt.xlabel('Epochs')
    plt.ylabel('acc')
    plt.legend()
    plt.show()

def get_callbacks():
    callbacks = [
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,  # Reduce learning rate by 50% when val_loss stops improving
            patience=2,
            min_lr=1e-4,
            verbose=1
        ),
        EarlyStopping(
            monitor='val_loss',
            patience=3,  # Stop training if no improvement after 10 epochs
            verbose=1,
            restore_best_weights=True
        ),
        ModelCheckpoint(
            filepath='best_model.keras',  # Save the best model during training
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        )
    ]
    return callbacks

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Input
from tensorflow.keras.layers import LSTM, Flatten, Dense

# Define LSTM model
input_tensor = Input(shape=(1, 8))
x = LSTM(units=64, return_sequences=True)(input_tensor)
x = LSTM(units=128, return_sequences=False)(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dense(32, activation='relu')(x)
output_tensor = Dense(8, activation='softmax')(x)

# Model initialization
model = models.Model(input_tensor, output_tensor)
opt = optimizers.Nadam(learning_rate=1e-3)
model.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()

batch_size = 512
epochs = 60
# Call_B_Fun = callbacks(Path,Dir)

history = model.fit(
    x_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=get_callbacks()
)

loss = history.history['loss']
val_loss = history.history['val_loss']
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
plot(loss,val_loss,acc,val_acc)

evaluation = model.evaluate(x_test,  y_test,batch_size=batch_size, verbose=2)
print()
print("Test loss :",evaluation[0]*100,"%")
print("Test accuracy :",evaluation[1]*100,"%")

# Define training function
@tf.function(input_signature=[tf.TensorSpec([None, 1, 8], tf.float32), tf.TensorSpec([None, 8], tf.float32)])
def train_step(x, y):
    with tf.GradientTape() as tape:
        prediction = model(x, training=True)
        loss = tf.reduce_mean(tf.keras.losses.categorical_crossentropy(y, prediction))
    gradients = tape.gradient(loss, model.trainable_variables)
    model.optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return {"loss": loss}

@tf.function(input_signature=[
    tf.TensorSpec(shape=[None, 1, 8], dtype=tf.float32, name="input")
])
def serve(input):
    input = (input - mean) / std
    return {"output": model(input)}

model.save('lstm_model.h5')
model.save('lstm_model.keras')

# Save the model with training and inference signatures
SAVED_MODEL_DIR = "my_saved_model"
tf.saved_model.save(
    model,
    SAVED_MODEL_DIR,
    signatures={
        "train": train_step,
        "serving_default": serve
    }
)

# Convert the model to TensorFlow Lite format
converter = tf.lite.TFLiteConverter.from_saved_model("my_saved_model")
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter.experimental_enable_resource_variables = True
tflite_model = converter.convert()

# Save TFLite model
with open('lstm_model_with_train_signature.tflite', 'wb') as f:
    f.write(tflite_model)


In [6]:
import tensorflow as tf

# Load the TFLite model
interpreter = tf.lite.Interpreter(model_path="/content/lstm_model_with_train_signature.tflite")

# Get the signature list
signatures = interpreter.get_signature_list()

# Print the signature list
print("Signature List:", signatures)

Signature List: {'serving_default': {'inputs': ['input'], 'outputs': ['output']}, 'train': {'inputs': ['x', 'y'], 'outputs': ['loss']}}
